#Creación de data dummie
##Instalación de dependencias

In [57]:
#Instalación de dependencias.
!pip install faker
import pandas as pd
import numpy as np
from faker import Faker
import random
import string

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.9 MB/s eta 0:00:00


In [58]:
#Configuración inicial para reproducibilidad de resultados.
fake = Faker('es_ES')
Faker.seed(42)
np.random.seed(42)
random.seed(42)

##Funciones para la generación de datos y errores

In [59]:
#Genera DNI de 8 dígitos con posibilidad de ceros a la izquierda.
def generate_dni():
    return str(random.randint(1, 99999999)).zfill(8)

In [60]:
#Inyecta espacios aleatorios al inicio, medio o final.
def inject_spaces(text):
    if pd.isna(text) or text == "": return text
    choice = random.choice(['start', 'middle', 'end', 'multiple'])
    if choice == 'start': return "  " + text
    if choice == 'end': return text + "   "
    if choice == 'middle':
        parts = text.split()
        if len(parts) > 1:
            return parts[0] + "    " + " ".join(parts[1:])
    return "  " + text + "  "

In [61]:
#Simula errores de dedo u omisión de letras.
def introduce_typos(text):
    if len(text) < 5: return text
    idx = random.randint(0, len(text) - 1)
    chars = list(text)
    if random.random() > 0.5:
        chars[idx] = random.choice(string.ascii_uppercase) # Sustitución
    else:
        chars.pop(idx) # Omisión
    return "".join(chars)

##Creación de la tabla 1: Lista de personal vigente
1000 registros base

In [62]:
#Generamos la Tabla 1
personal_data = []
for i in range(1, 1001):
    personal_data.append({
        'N°': i,
        'DNI': generate_dni(),
        'APELLIDOS': fake.last_name().upper() + " " + fake.last_name().upper(),
        'NOMBRES': fake.first_name().upper() + " " + fake.first_name().upper(),
        'CARGO': random.choice(['Analista G1', 'Especialista Técnico', 'Coordinador Adjunto', 'Asistente Administrativo']),
        'FECHA DE INICIO': fake.date_between(start_date='-10y', end_date='today'),
        'REGIMEN LABORAL': random.choice(['CAS', 'D.L. 728', 'D.L. 276']),
        'ÓRGANO': 'Dirección General de ' + fake.word().capitalize(),
        'UNIDAD ORGANICA': 'Oficina de ' + fake.word().capitalize(),
        'ESTADO': 'ACTIVO',
        'SERVIDORES CIVILES QUE TIENEN LICENCIA': 'NO',
        'CORREO ELECTRONICO INSTITUCIONAL': np.nan
    })

df1 = pd.DataFrame(personal_data)

In [63]:
# Inyectar espacios (10%)
idx_spaces = df1.sample(frac=0.1).index
df1.loc[idx_spaces, 'APELLIDOS'] = df1.loc[idx_spaces, 'APELLIDOS'].apply(inject_spaces)
df1.loc[idx_spaces, 'NOMBRES'] = df1.loc[idx_spaces, 'NOMBRES'].apply(inject_spaces)

In [64]:
#Se agrega la columna APELLIDOS Y NOMBRES
df1.loc[:, 'APELLIDOS Y NOMBRES'] = df1['APELLIDOS'] + ' ' + df1['NOMBRES']

In [65]:
# Inyectar duplicados de puesto (5%)
duplicados = df1.sample(n=50).copy()
duplicados['CARGO'] = 'Puesto Secundario'
df1 = pd.concat([df1, duplicados], ignore_index=True)

##Creación de la tabla 2: Lista de correos institucionales
1300 registros

In [66]:
#Generamos la Tabla 2
ws_data = []

# 50% de coincidencia con Tabla 1
t1_sample = df1.drop_duplicates('DNI').sample(n=650)
exact_match_n = int(650 * 0.6)

# Obtenemos la posición de las columnas NOMBRES y APELLIDOS en t1_sample
posicion_col_apellidos = t1_sample.columns.get_loc("APELLIDOS")
posicion_col_nombres = t1_sample.columns.get_loc("NOMBRES")

for i, pandas_tuple in enumerate(t1_sample.itertuples(index=False)):
    last_n = pandas_tuple[posicion_col_apellidos]
    first_n = pandas_tuple[posicion_col_nombres]

    if i >= exact_match_n: # Coincidencia aproximada (40%)
        last_n = introduce_typos(last_n)

    # Generar correo dummie
    apellidos = last_n.split()
    email = (first_n[0] + apellidos[0] + "@dummie.com").lower()

    ws_data.append({
        'First Name [Required]': first_n,
        'Last Name [Required]': last_n,
        'Email Address [Required]': email,
        'Status [READ ONLY]': 'ACTIVE',
        'Last Sign In [READ ONLY]': fake.date_time_this_year(),
        'Storage Used [READ ONLY]': f"{random.randint(1, 500)} MB"
    })

In [67]:
# Resto de registros (cuentas funcionales y otros)
for _ in range(650):
    is_functional = random.random() < 0.3
    f_name = "" if is_functional else fake.first_name().upper()
    l_name = fake.word().upper() if is_functional else fake.last_name().upper()
    email =  f"{l_name.lower()}@dummie.com" if is_functional else (f_name[0] + l_name + "@dummie.com").lower()

    ws_data.append({
        'First Name [Required]': f_name,
        'Last Name [Required]': l_name,
        'Email Address [Required]': email,
        'Status [READ ONLY]': 'ACTIVE',
        'Last Sign In [READ ONLY]': fake.date_time_this_year(),
        'Storage Used [READ ONLY]': '0 MB'
    })

df2 = pd.DataFrame(ws_data)

In [68]:
# Inyectar duplicados de nombre con distinto correo (10%)
dup_ws = df2.sample(frac=0.1).copy()
dup_ws['Email Address [Required]'] = dup_ws['Email Address [Required]'].apply(lambda x: "alt_" + x)
df2 = pd.concat([df2, dup_ws], ignore_index=True)

##Creación de la tabla 3: Lista de cuentas de directorio activo
1200 registros

# Tomar muestra de T1 para Description
t1_pool = df1['DNI'].unique().tolist()
non_empty_count = int(1200 * 0.9)

for i in range(1200):
    desc = np.nan
    f_name = fake.first_name().upper()
    l_name = fake.last_name().upper()
    
    if i < non_empty_count:
        rand_val = random.random()
        if rand_val < 0.6: # DNI Real
            desc = random.choice(t1_pool)
        elif rand_val < 0.8: # DNI Erróneo (más/menos dígitos)
            desc = str(random.randint(1, 9999999999))[:random.choice([7, 9, 10])]
        else: # Igual a First Name
            desc = f_name
            
    ad_data.append({
        'Name': f"{l_name} {f_name}",
        'Description': desc,
        'Disabled': 'FALSE',
        'Email Address': f"{f_name[0].lower()}{l_name.lower()}@dummie.com",
        'First Name': f_name,
        'Last Name': l_name,
        'Username (pre 2000)': f"DUMMIE\\{f_name[0]}{l_name}"[:15]
    })

df3 = pd.DataFrame(ad_data)

In [70]:
#Generamos la tabla 3
ad_data = []

# Tomar muestra de T1 para Description
non_empty_count = int(1200 * 0.65)
t1_sample_2 = df1.drop_duplicates('DNI').sample(non_empty_count)

# Obtenemos la posición de las columnas NOMBRES, APELLIDOS, DNI en t1_sample_2
posicion_col_apellidos = t1_sample_2.columns.get_loc("APELLIDOS")
posicion_col_nombres = t1_sample_2.columns.get_loc("NOMBRES")
posicion_col_dni = t1_sample_2.columns.get_loc("DNI")

for i, pandas_tuple in enumerate(t1_sample_2.itertuples(index=False)):
    last_n = pandas_tuple[posicion_col_apellidos]
    first_n = pandas_tuple[posicion_col_nombres]

    # Generar correo dummie
    apellidos = last_n.split()
    email = (first_n[0] + apellidos[0] + "@dummie.com").lower()

    rand_val = random.random()
    if rand_val < 0.6: # DNI Real
        desc = pandas_tuple[posicion_col_dni]

    elif rand_val < 0.8: # DNI Erróneo (más/menos dígitos)
        desc = str(pandas_tuple[posicion_col_dni]).zfill(10)[:random.choice([7, 9, 10])]
    else: # Igual a First Name
        desc = f_name

    ad_data.append({
        'Name': f"{last_n} {first_n}",
        'Description': desc,
        'Disabled': 'FALSE',
        'Email Address': email,
        'First Name': first_n,
        'Last Name': last_n,
        'Username (pre 2000)': f"DUMMIE\\{first_n[0]}{apellidos[0]}"[:15]
    })


In [72]:
# Resto de registros
empty_count = int(1200 * 0.35)
for _ in range(empty_count):
    desc = np.nan
    f_name = fake.first_name().upper()
    l_name = fake.last_name().upper()
    email =  (f_name[0] + l_name + "@dummie.com").lower()

    ad_data.append({
        'Name': f"{l_name} {f_name}",
        'Description': desc,
        'Disabled': 'TRUE',
        'Email Address': email,
        'First Name': f_name,
        'Last Name': l_name,
        'Username (pre 2000)': f"DUMMIE\\{f_name}{l_name}"[:15]
    })

df3 = pd.DataFrame(ad_data)

In [73]:
# Inyectar duplicados en AD (10%)
dup_ad = df3.sample(frac=0.1).copy()
dup_ad['Email Address'] = dup_ad['Email Address'].apply(lambda x: "2_" + x)
df3 = pd.concat([df3, dup_ad], ignore_index=True)

##Conteo de registros y exportación de datos

In [74]:
print("\nDataframes generados con éxito:")
print(f"- tabla1_personal.csv: {len(df1)} registros")
print(f"- tabla2_workspace.csv: {len(df2)} registros")
print(f"- tabla3_directorio.csv: {len(df3)} registros")


Dataframes generados con éxito:
- tabla1_personal.csv: 1050 registros
- tabla2_workspace.csv: 1430 registros
- tabla3_directorio.csv: 1320 registros


In [75]:
df1.to_csv('tabla1_personal.csv', index=False, encoding='utf-8-sig')
df2.to_csv('tabla2_workspace.csv', index=False, encoding='utf-8-sig')
df3.to_csv('tabla3_directorio.csv', index=False, encoding='utf-8-sig')